In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

focal_df = pd.read_csv(
    PROCESSED_DIR / "focal_2016_2025.csv"
)

dev_df = pd.read_csv(
    PROCESSED_DIR / "development_2016_2025.csv"
)

print("Focal:", focal_df.shape)
print("Development:", dev_df.shape)

Focal: (180, 10)
Development: (1755, 8)


In [3]:
print("Focal columns:")
print(focal_df.columns.tolist())

print("\nDevelopment columns:")
print(dev_df.columns.tolist())

Focal columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'filepath', 'text', 'word_count', 'country', 'region']

Development columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'filepath', 'text', 'word_count']


In [4]:
print("Missing focal text:", focal_df["text"].isna().sum())
print("Missing development text:", dev_df["text"].isna().sum())

print("Empty focal text:",
      (focal_df["text"].fillna("").str.strip() == "").sum())

print("Empty development text:",
      (dev_df["text"].fillna("").str.strip() == "").sum())

Missing focal text: 0
Missing development text: 0
Empty focal text: 0
Empty development text: 0


In [5]:
focal_df["text_clean"] = focal_df["text"].copy()
dev_df["text_clean"] = dev_df["text"].copy()

In [6]:
def normalize_unicode_and_whitespace(text):
    if pd.isna(text):
        return ""

    # Normalize Unicode without removing meaningful characters
    text = unicodedata.normalize("NFC", text)

    # Replace unusual Unicode line/paragraph separators
    text = text.replace("\u2028", "\n")
    text = text.replace("\u2029", "\n")

    # Replace non-breaking spaces with normal spaces
    text = text.replace("\u00A0", " ")

    # Standardize different newline formats
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Convert tabs to spaces
    text = text.replace("\t", " ")

    # Collapse repeated spaces within lines
    text = re.sub(r"[ ]{2,}", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [7]:
focal_df["text_clean"] = focal_df["text"].apply(
    normalize_unicode_and_whitespace
)

dev_df["text_clean"] = dev_df["text"].apply(
    normalize_unicode_and_whitespace
)

In [8]:
print("Empty focal cleaned text:",
      (focal_df["text_clean"].str.strip() == "").sum())

print("Empty development cleaned text:",
      (dev_df["text_clean"].str.strip() == "").sum())

Empty focal cleaned text: 0
Empty development cleaned text: 0


In [9]:
focal_df["original_chars"] = focal_df["text"].str.len()
focal_df["clean_chars"] = focal_df["text_clean"].str.len()

focal_df[
    ["iso3", "year", "original_chars", "clean_chars"]
].head(10)

,iso3,year,original_chars,clean_chars
0,CHL,2016,8039,8038
1,CHL,2017,15549,15549
2,CHL,2018,26641,26558
3,CHL,2019,16335,16334
4,CHL,2020,19274,19274
5,CHL,2021,18698,18697
6,CHL,2022,14946,14945
7,CHL,2023,17065,17064
8,CHL,2024,12437,12435
9,CHL,2025,16946,16946


In [10]:
print("ORIGINAL:")
print(focal_df.loc[0, "text"][:1500])

print("\n" + "=" * 80 + "\n")

print("CLEANED:")
print(focal_df.loc[0, "text_clean"][:1500])

ORIGINAL:
We are living in times that show us the necessity of shifting our perspective with regard to addressing development. Our world is in the midst of a crisis that extends back years, and which, far from being resolved, has only worsened, generating criticism of political institutions that do not seem capable of representing their citizens. At the same time, economic growth has given way to a global slowdown and has not been able to provide the world’s population with the well-being that it seeks.
It would appear that the far-reaching dynamics that caused the crisis some years ago were not superseded but merely superficially retouched. But citizens today are more aware and empowered. Thanks to new technology, they have new means of expression. They are showing us that in their daily lives, at their places of work, at their children’s schools, in the health of their families, in the security of their cities, in the pensions of their mothers and fathers and in the relations between

In [11]:
focal_df["text_lower"] = focal_df["text_clean"].str.lower()
dev_df["text_lower"] = dev_df["text_clean"].str.lower()

In [12]:
focal_df["word_count_clean"] = (
    focal_df["text_clean"]
    .str.split()
    .str.len()
)

dev_df["word_count_clean"] = (
    dev_df["text_clean"]
    .str.split()
    .str.len()
)

In [13]:
focal_df["word_count_clean"].describe()

count     180.000000
mean     2171.383333
std       951.973952
min       611.000000
25%      1554.000000
50%      1978.500000
75%      2571.750000
max      8242.000000
Name: word_count_clean, dtype: float64

In [14]:
dev_df["word_count_clean"].describe()

count    1755.000000
mean     2037.878063
std       702.201454
min       463.000000
25%      1580.500000
50%      1939.000000
75%      2398.000000
max      7082.000000
Name: word_count_clean, dtype: float64

In [15]:
print(
    "Total focal words:",
    focal_df["word_count_clean"].sum()
)

print(
    "Total development words:",
    dev_df["word_count_clean"].sum()
)

Total focal words: 390849
Total development words: 3576476


In [16]:
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
focal_df.to_csv(
    PREPROCESSED_DIR / "focal_preprocessed.csv",
    index=False
)

dev_df.to_csv(
    PREPROCESSED_DIR / "development_preprocessed.csv",
    index=False
)

In [18]:
focal_df[
    ["iso3", "year", "filename", "word_count_clean"]
].sort_values("word_count_clean").head(5)

,iso3,year,filename,word_count_clean
55,IDN,2021,IDN_76_2021.txt,611
74,JOR,2020,JOR_75_2020.txt,738
75,JOR,2021,JOR_76_2021.txt,903
54,IDN,2020,IDN_75_2020.txt,950
174,USA,2020,USA_75_2020.txt,981


In [19]:
dev_df[
    ["iso3", "year", "filename", "word_count_clean"]
].sort_values("word_count_clean").head(5)

,iso3,year,filename,word_count_clean
13,BEN,2016,BEN_71_2016.txt,463
945,GNB,2021,GNB_76_2021.txt,480
1017,RWA,2021,RWA_76_2021.txt,492
447,LTU,2018,LTU_73_2018.txt,508
269,LTU,2017,LTU_72_2017.txt,537


In [20]:
shortest_idx = focal_df["word_count_clean"].idxmin()

print(
    focal_df.loc[
        shortest_idx,
        ["iso3", "year", "filename", "word_count_clean"]
    ]
)

print("\n--- BEGINNING OF SPEECH ---\n")
print(focal_df.loc[shortest_idx, "text_clean"][:2000])

iso3                            IDN
year                           2021
filename            IDN_76_2021.txt
word_count_clean                611
Name: 55, dtype: object

--- BEGINNING OF SPEECH ---

His Excellency President of the UN General Assembly,
His Excellency UN Secretary-General,
Excellencies, Leaders of UN Member Countries.
The world community looks towards the UN General Assembly to provide answers to major global concerns.
When will the people be free from the pandemic? When will the economy recover and grow inclusively? How to sustain the planet for future generations? And when will the world be free from conflict, terrorism, and war?
Given the recent global developments, there are many things that we need to do together.
First, we must bring hope that we can tackle the COVID-19 pandemic swiftly, fairly, and equally.
We know that “no one is safe until everyone is”. The capacity and pace among countries in tackling COVID-19, including in vaccinations, differs widely. Politici

In [21]:
import spacy

In [22]:
nlp = spacy.load("en_core_web_sm")

In [23]:
nlp.max_length = 2_000_000

In [24]:
def segment_speech(text):
    doc = nlp(text)

    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if sent.text.strip()
    ]

    return sentences

In [25]:
test_text = focal_df.loc[0, "text_clean"]

test_sentences = segment_speech(test_text)

print("Number of sentences:", len(test_sentences))

for i, sentence in enumerate(test_sentences[:20]):
    print(i, sentence)

Number of sentences: 64
0 We are living in times that show us the necessity of shifting our perspective with regard to addressing development.
1 Our world is in the midst of a crisis that extends back years, and which, far from being resolved, has only worsened, generating criticism of political institutions that do not seem capable of representing their citizens.
2 At the same time, economic growth has given way to a global slowdown and has not been able to provide the world’s population with the well-being that it seeks.
3 It would appear that the far-reaching dynamics that caused the crisis some years ago were not superseded but merely superficially retouched.
4 But citizens today are more aware and empowered.
5 Thanks to new technology, they have new means of expression.
6 They are showing us that in their daily lives, at their places of work, at their children’s schools, in the health of their families, in the security of their cities, in the pensions of their mothers and fathers 

In [26]:
def build_sentence_dataframe(df):
    sentence_records = []

    for _, row in df.iterrows():
        sentences = segment_speech(row["text_clean"])

        for sentence_id, sentence in enumerate(sentences):
            sentence_records.append({
                "corpus": row["corpus"],
                "iso3": row["iso3"],
                "session": row["session"],
                "year": row["year"],
                "filename": row["filename"],
                "sentence_id": sentence_id,
                "sentence_text": sentence,
                "sentence_text_lower": sentence.lower()
            })

    return pd.DataFrame(sentence_records)

In [27]:
focal_sent_df = build_sentence_dataframe(focal_df)

In [28]:
print("Focal sentence dataframe shape:", focal_sent_df.shape)

Focal sentence dataframe shape: (18903, 8)


In [29]:
focal_sent_df.head(10)

,corpus,iso3,session,year,filename,sentence_id,sentence_text,sentence_text_lower
0,focal,CHL,71,2016,CHL_71_2016.txt,0,We are living in times that show us the necess...,we are living in times that show us the necess...
1,focal,CHL,71,2016,CHL_71_2016.txt,1,Our world is in the midst of a crisis that ext...,our world is in the midst of a crisis that ext...
2,focal,CHL,71,2016,CHL_71_2016.txt,2,"At the same time, economic growth has given wa...","at the same time, economic growth has given wa..."
3,focal,CHL,71,2016,CHL_71_2016.txt,3,It would appear that the far-reaching dynamics...,it would appear that the far-reaching dynamics...
4,focal,CHL,71,2016,CHL_71_2016.txt,4,But citizens today are more aware and empowered.,but citizens today are more aware and empowered.
5,focal,CHL,71,2016,CHL_71_2016.txt,5,"Thanks to new technology, they have new means ...","thanks to new technology, they have new means ..."
6,focal,CHL,71,2016,CHL_71_2016.txt,6,"They are showing us that in their daily lives,...","they are showing us that in their daily lives,..."
7,focal,CHL,71,2016,CHL_71_2016.txt,7,This is the main basis for the citizen discont...,this is the main basis for the citizen discont...
8,focal,CHL,71,2016,CHL_71_2016.txt,8,"At the same time, we are facing an unprecedent...","at the same time, we are facing an unprecedent..."
9,focal,CHL,71,2016,CHL_71_2016.txt,9,These are combined with violence and armed con...,these are combined with violence and armed con...


In [30]:
dev_sent_df = build_sentence_dataframe(dev_df)

In [31]:
print("Development sentence dataframe shape:", dev_sent_df.shape)

Development sentence dataframe shape: (157525, 8)


In [32]:
dev_sent_df.head(10)

,corpus,iso3,session,year,filename,sentence_id,sentence_text,sentence_text_lower
0,development,AFG,71,2016,AFG_71_2016.txt,0,It is my distinct pleasure to attend the seven...,it is my distinct pleasure to attend the seven...
1,development,AFG,71,2016,AFG_71_2016.txt,1,I sincerely congratulate Ambassador Peter Thom...,i sincerely congratulate ambassador peter thom...
2,development,AFG,71,2016,AFG_71_2016.txt,2,I thank Secretary-General Ban Ki-moon for his ...,i thank secretary-general ban ki-moon for his ...
3,development,AFG,71,2016,AFG_71_2016.txt,3,We wish him every success in his future endeav...,we wish him every success in his future endeav...
4,development,AFG,71,2016,AFG_71_2016.txt,4,The twenty-first century has brought with it a...,the twenty-first century has brought with it a...
5,development,AFG,71,2016,AFG_71_2016.txt,5,We have come to realize that dialogue and coop...,we have come to realize that dialogue and coop...
6,development,AFG,71,2016,AFG_71_2016.txt,6,We believe that the United Nations remains the...,we believe that the united nations remains the...
7,development,AFG,71,2016,AFG_71_2016.txt,7,The United Nations vital role in helping Afgha...,the united nations vital role in helping afgha...
8,development,AFG,71,2016,AFG_71_2016.txt,8,Allow me to update the Assembly on the current...,allow me to update the assembly on the current...
9,development,AFG,71,2016,AFG_71_2016.txt,9,While two years have passed since the establis...,while two years have passed since the establis...


In [33]:
print("Total focal sentences:", len(focal_sent_df))
print("Total development sentences:", len(dev_sent_df))

Total focal sentences: 18903
Total development sentences: 157525


In [34]:
focal_sentence_counts = (
    focal_sent_df
    .groupby("filename")
    .size()
)

dev_sentence_counts = (
    dev_sent_df
    .groupby("filename")
    .size()
)

print("\nFocal sentences per speech:")
print(focal_sentence_counts.describe())

print("\nDevelopment sentences per speech:")
print(dev_sentence_counts.describe())


Focal sentences per speech:
count    180.000000
mean     105.016667
std       58.792835
min       31.000000
25%       73.750000
50%       93.000000
75%      125.000000
max      635.000000
dtype: float64

Development sentences per speech:
count    1755.000000
mean       89.757835
std        36.890590
min        14.000000
25%        66.000000
50%        84.000000
75%       106.000000
max       388.000000
dtype: float64


In [35]:
idn_2021 = focal_sent_df[
    (focal_sent_df["iso3"] == "IDN") &
    (focal_sent_df["year"] == 2021)
]

idn_2021[
    ["sentence_id", "sentence_text"]
].head(20)

,sentence_id,sentence_text
5888,0,His Excellency President of the UN General Ass...
5889,1,The world community looks towards the UN Gener...
5890,2,When will the people be free from the pandemic?
5891,3,When will the economy recover and grow inclusi...
5892,4,How to sustain the planet for future generations?
5893,5,"And when will the world be free from conflict,..."
5894,6,"Given the recent global developments, there ar..."
5895,7,"First, we must bring hope that we can tackle t..."
5896,8,We know that “no one is safe until everyone is”.
5897,9,The capacity and pace among countries in tackl...


In [36]:
print(
    "Empty focal sentences:",
    (focal_sent_df["sentence_text"].str.strip() == "").sum()
)

print(
    "Empty development sentences:",
    (dev_sent_df["sentence_text"].str.strip() == "").sum()
)

Empty focal sentences: 0
Empty development sentences: 0


In [37]:
focal_sent_df.to_csv(
    PREPROCESSED_DIR / "focal_sentences.csv",
    index=False
)

dev_sent_df.to_csv(
    PREPROCESSED_DIR / "development_sentences.csv",
    index=False
)

In [38]:
focal_sentence_counts.sort_values(ascending=False).head(10)

filename
USA_80_2025.txt    635
PAK_74_2019.txt    346
USA_71_2016.txt    235
COL_80_2025.txt    222
USA_79_2024.txt    219
USA_72_2017.txt    208
USA_74_2019.txt    203
USA_73_2018.txt    199
USA_78_2023.txt    185
POL_78_2023.txt    180
dtype: int64

In [39]:
max_sentence_file = focal_sentence_counts.idxmax()

print("Speech with most sentences:", max_sentence_file)
print("Number of sentences:", focal_sentence_counts.loc[max_sentence_file])

Speech with most sentences: USA_80_2025.txt
Number of sentences: 635


In [40]:
max_sentence_speech = focal_sent_df[
    focal_sent_df["filename"] == max_sentence_file
]

max_sentence_speech[
    ["sentence_id", "sentence_text"]
].head(30)

,sentence_id,sentence_text
18268,0,The Assembly will hear an address by His Excel...
18269,1,I request protocol to escort His Excellency an...
18270,2,"Thank you very much, very much appreciate it."
18271,3,And I don't mind making this speech without a ...
18272,4,"I feel very happy to be up here with you, neve..."
18273,5,I can only say that whoever is operating this ...
18274,6,"Hello, Madam First Lady."
18275,7,"Thank you very much for being here, and Madam ..."
18276,8,Six years have passed since I last stood in th...
18277,9,"Since that day, the guns of war have shattered..."


In [41]:
dev_sentence_counts.sort_values().head(10)

filename
GNB_76_2021.txt    14
BEN_71_2016.txt    19
ERI_75_2020.txt    23
RWA_72_2017.txt    25
ZMB_75_2020.txt    25
AGO_73_2018.txt    25
LTU_73_2018.txt    25
BHR_75_2020.txt    25
GNQ_77_2022.txt    26
GIN_75_2020.txt    27
dtype: int64

In [42]:
min_sentence_file = dev_sentence_counts.idxmin()

print("Speech with fewest sentences:", min_sentence_file)
print("Number of sentences:", dev_sentence_counts.loc[min_sentence_file])

Speech with fewest sentences: GNB_76_2021.txt
Number of sentences: 14


In [43]:
min_sentence_speech = dev_sent_df[
    dev_sent_df["filename"] == min_sentence_file
]

for _, row in min_sentence_speech.head(14).iterrows():
    print(row["sentence_id"], row["sentence_text"])
    print()

0 I warmly congratulate the Minister for Foreign Affairs of the Maldives, Mr. Abdulla Shahid, President of the General Assembly, and wish him every success.

1 As we face the challenges brought about by the coronavirus disease (COVID-19) pandemic, the theme chosen for this year’s general debate could not be more pertinent.

2 Only with renewed hope will we be able to build resilience to overcome the socioeconomic and humanitarian difficulties caused by the worldwide health crisis and to adequately and sustainably respond to the current challenges.

3 In an increasingly globalized world, the relevance of the existence of the United Nations and its agencies has become of the utmost importance.

4 It is therefore urgent to improve the efficiency of the Organization and adopt the necessary reforms of all its organs, including the Security Council, with a view to revitalizing the United Nations system in its entirety.

5 That is the only way for us to work better towards the implementation 

In [44]:
first_sentences = (
    pd.concat([focal_sent_df, dev_sent_df], ignore_index=True)
    .query("sentence_id < 5")
)

print("Number of first-five sentence records:", len(first_sentences))

Number of first-five sentence records: 9675


In [45]:
assembly_intro = first_sentences[
    first_sentences["sentence_text_lower"].str.contains(
        "the assembly will hear an address",
        na=False
    )
]

print("Speeches containing this procedural introduction:",
      assembly_intro["filename"].nunique())

assembly_intro[
    ["iso3", "year", "filename", "sentence_id", "sentence_text"]
].head(30)

Speeches containing this procedural introduction: 75


,iso3,year,filename,sentence_id,sentence_text
1038,CHL,2025,CHL_80_2025.txt,0,The Assembly will hear an address by His Excel...
3249,COL,2025,COL_80_2025.txt,0,The Assembly will hear an address by His Excel...
5315,GHA,2025,GHA_80_2025.txt,0,The Assembly will hear an address by His Excel...
6177,IDN,2025,IDN_80_2025.txt,1,The Assembly will hear an address by His Excel...
7931,JOR,2025,JOR_80_2025.txt,0,The Assembly will hear an address by His Majes...
9929,KAZ,2025,KAZ_80_2025.txt,0,The Assembly will hear an address by His Excel...
11034,KEN,2025,KEN_80_2025.txt,0,The Assembly will hear an address by His Excel...
18268,USA,2025,USA_80_2025.txt,0,The Assembly will hear an address by His Excel...
158331,AGO,2025,AGO_80_2025.txt,0,The Assembly will hear an address by His Excel...
158782,ARM,2025,ARM_80_2025.txt,0,The Assembly will hear an address by His Excel...


In [46]:
protocol_intro = first_sentences[
    first_sentences["sentence_text_lower"].str.contains(
        "i request protocol",
        na=False
    )
]

print("Speeches containing protocol-introduction language:",
      protocol_intro["filename"].nunique())

protocol_intro[
    ["iso3", "year", "filename", "sentence_id", "sentence_text"]
].head(30)

Speeches containing protocol-introduction language: 117


,iso3,year,filename,sentence_id,sentence_text
1039,CHL,2025,CHL_80_2025.txt,1,I request protocol to escort His Excellency an...
2236,CHN,2025,CHN_80_2025.txt,1,I request Protocol to escort His Excellency an...
3250,COL,2025,COL_80_2025.txt,1,I request protocol to escort His Excellency an...
5316,GHA,2025,GHA_80_2025.txt,1,I request protocol to escort His Excellency an...
6178,IDN,2025,IDN_80_2025.txt,2,I request protocol to escort His Excellency an...
7932,JOR,2025,JOR_80_2025.txt,1,I request protocol to escort His Majesty and i...
8842,JPN,2025,JPN_80_2025.txt,1,I request protocol to escort His Excellency an...
9930,KAZ,2025,KAZ_80_2025.txt,1,I request protocol to escort His Excellency an...
11035,KEN,2025,KEN_80_2025.txt,1,I request protocol to escort His Excellency an...
12485,PAK,2025,PAK_80_2025.txt,1,I request protocol to escort His Excellency an...


In [47]:
for phrase in [
    "the assembly will hear an address",
    "i request protocol",
    "may i request protocol",
    "escort his excellency",
    "escort her excellency"
]:
    matches = first_sentences[
        first_sentences["sentence_text_lower"].str.contains(
            phrase,
            na=False,
            regex=False
        )
    ]

    print(
        phrase,
        "->",
        matches["filename"].nunique(),
        "speeches"
    )

the assembly will hear an address -> 75 speeches
i request protocol -> 117 speeches
may i request protocol -> 0 speeches
escort his excellency -> 104 speeches
escort her excellency -> 10 speeches


In [48]:
PROCEDURAL_PATTERNS = [
    "the assembly will hear an address",
    "i request protocol",
    "escort his excellency",
    "escort her excellency"
]

In [49]:
def is_procedural_intro(sentence, sentence_id):
    if sentence_id >= 5:
        return False

    sentence_lower = sentence.lower()

    return any(
        pattern in sentence_lower
        for pattern in PROCEDURAL_PATTERNS
    )

In [50]:
sentence_lower = sentence.lower()

In [51]:
any(
    pattern in sentence_lower
    for pattern in PROCEDURAL_PATTERNS
)

False

In [52]:
focal_sent_df["procedural_intro"] = focal_sent_df.apply(
    lambda row: is_procedural_intro(
        row["sentence_text"],
        row["sentence_id"]
    ),
    axis=1
)

dev_sent_df["procedural_intro"] = dev_sent_df.apply(
    lambda row: is_procedural_intro(
        row["sentence_text"],
        row["sentence_id"]
    ),
    axis=1
)

In [53]:
print(
    "Flagged focal procedural sentences:",
    focal_sent_df["procedural_intro"].sum()
)

print(
    "Flagged development procedural sentences:",
    dev_sent_df["procedural_intro"].sum()
)

Flagged focal procedural sentences: 21
Flagged development procedural sentences: 179


In [54]:
print(
    "Affected focal speeches:",
    focal_sent_df.loc[
        focal_sent_df["procedural_intro"],
        "filename"
    ].nunique()
)

print(
    "Affected development speeches:",
    dev_sent_df.loc[
        dev_sent_df["procedural_intro"],
        "filename"
    ].nunique()
)

Affected focal speeches: 13
Affected development speeches: 115


In [55]:
focal_sent_df.loc[
    focal_sent_df["procedural_intro"],
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "sentence_text"
    ]
].head(50)

,iso3,year,filename,sentence_id,sentence_text
1038,CHL,2025,CHL_80_2025.txt,0,The Assembly will hear an address by His Excel...
1039,CHL,2025,CHL_80_2025.txt,1,I request protocol to escort His Excellency an...
2236,CHN,2025,CHN_80_2025.txt,1,I request Protocol to escort His Excellency an...
3249,COL,2025,COL_80_2025.txt,0,The Assembly will hear an address by His Excel...
3250,COL,2025,COL_80_2025.txt,1,I request protocol to escort His Excellency an...
5315,GHA,2025,GHA_80_2025.txt,0,The Assembly will hear an address by His Excel...
5316,GHA,2025,GHA_80_2025.txt,1,I request protocol to escort His Excellency an...
6177,IDN,2025,IDN_80_2025.txt,1,The Assembly will hear an address by His Excel...
6178,IDN,2025,IDN_80_2025.txt,2,I request protocol to escort His Excellency an...
7931,JOR,2025,JOR_80_2025.txt,0,The Assembly will hear an address by His Majes...


In [56]:
dev_sent_df.loc[
    dev_sent_df["procedural_intro"],
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "sentence_text"
    ]
].head(50)

,iso3,year,filename,sentence_id,sentence_text
139428,AGO,2025,AGO_80_2025.txt,0,The Assembly will hear an address by His Excel...
139429,AGO,2025,AGO_80_2025.txt,1,I request protocol to escort His Excellency an...
139484,ALB,2025,ALB_80_2025.txt,1,I request Protocol to escort His Excellency an...
139556,AND,2025,AND_80_2025.txt,1,I request Protocol to escort His Excellency an...
139879,ARM,2025,ARM_80_2025.txt,0,The Assembly will hear an address by His Excel...
139880,ARM,2025,ARM_80_2025.txt,1,I request protocol to escort His Excellency an...
139985,ATG,2025,ATG_80_2025.txt,0,"And now, the Assembly will hear an address by ..."
140109,AUS,2025,AUS_80_2025.txt,0,Now the Assembly will hear an address by His E...
140110,AUS,2025,AUS_80_2025.txt,1,And I request the protocol to escort His Excel...
140337,AZE,2025,AZE_80_2025.txt,0,The Assembly will hear an address by His Excel...


In [57]:
print(
    focal_sent_df.loc[
        focal_sent_df["procedural_intro"],
        "sentence_id"
    ].value_counts().sort_index()
)

print(
    dev_sent_df.loc[
        dev_sent_df["procedural_intro"],
        "sentence_id"
    ].value_counts().sort_index()
)

sentence_id
0     7
1    13
2     1
Name: count, dtype: int64
sentence_id
0     69
1    107
2      3
Name: count, dtype: int64


In [58]:
focal_sent_clean_df = focal_sent_df[
    ~focal_sent_df["procedural_intro"]
].copy()

dev_sent_clean_df = dev_sent_df[
    ~dev_sent_df["procedural_intro"]
].copy()

In [59]:
focal_sent_clean_df["sentence_id"] = (
    focal_sent_clean_df
    .groupby("filename")
    .cumcount()
)

dev_sent_clean_df["sentence_id"] = (
    dev_sent_clean_df
    .groupby("filename")
    .cumcount()
)

In [60]:
print("Original focal sentences:", len(focal_sent_df))
print("Clean focal sentences:", len(focal_sent_clean_df))

print("Original development sentences:", len(dev_sent_df))
print("Clean development sentences:", len(dev_sent_clean_df))

Original focal sentences: 18903
Clean focal sentences: 18882
Original development sentences: 157525
Clean development sentences: 157346


In [61]:
print(
    "Remaining focal procedural flags:",
    focal_sent_clean_df["procedural_intro"].sum()
)

print(
    "Remaining development procedural flags:",
    dev_sent_clean_df["procedural_intro"].sum()
)

Remaining focal procedural flags: 0
Remaining development procedural flags: 0


In [62]:
focal_sent_clean_df[
    focal_sent_clean_df["filename"] == "USA_80_2025.txt"
][["sentence_id", "sentence_text"]].head(10)

,sentence_id,sentence_text
18270,0,"Thank you very much, very much appreciate it."
18271,1,And I don't mind making this speech without a ...
18272,2,"I feel very happy to be up here with you, neve..."
18273,3,I can only say that whoever is operating this ...
18274,4,"Hello, Madam First Lady."
18275,5,"Thank you very much for being here, and Madam ..."
18276,6,Six years have passed since I last stood in th...
18277,7,"Since that day, the guns of war have shattered..."
18278,8,An era of calm and stability gave way to one o...
18279,9,"And here in the United States, four years of w..."


In [63]:
focal_sent_clean_df.to_csv(
    PREPROCESSED_DIR / "focal_sentences_clean.csv",
    index=False
)

dev_sent_clean_df.to_csv(
    PREPROCESSED_DIR / "development_sentences_clean.csv",
    index=False
)

In [64]:
print(
    "Remaining focal procedural flags:",
    focal_sent_clean_df["procedural_intro"].sum()
)

print(
    "Remaining development procedural flags:",
    dev_sent_clean_df["procedural_intro"].sum()
)

Remaining focal procedural flags: 0
Remaining development procedural flags: 0
